# Gradient Descent Variants — From Scratch

**Author:** Siddharth Bokka  
**Surface:** Quadratic (convex), Rosenbrock (non-convex), Himmelblau (multi-modal)  
**Framework:** Pure NumPy — no PyTorch, no autograd  

---

## What This Notebook Covers

| Algorithm | Key Property | When It Wins |
|---|---|---|
| Batch Gradient Descent | Full dataset per step | Small datasets, convex problems |
| Stochastic GD (SGD) | 1 sample per step | Escaping local minima, online learning |
| Mini-Batch GD | B samples per step | **Industry standard** — all modern DL |
| SGD + Momentum | Velocity accumulation | Ill-conditioned surfaces, oscillation damping |

All four algorithms are implemented from scratch in `gdo.optimizers`  
and run on analytical loss surfaces so every trajectory is exact.

---

**Key questions answered:**
- Why does vanilla GD oscillate on elongated loss surfaces?
- Why does SGD escape local minima but not converge cleanly?
- Why does momentum dampen oscillations?
- What does the optimization trajectory look like on the Rosenbrock banana?

## Step 1 — Setup

In [ ]:
import sys
import logging
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib

# Make src/ importable without installing
repo_root = Path().resolve().parent
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

# Logging: INFO level so we see convergence messages
logging.basicConfig(
    format='%(asctime)s | %(levelname)-8s | %(name)s | %(message)s',
    datefmt='%H:%M:%S',
    level=logging.INFO,
)

# Plot style
matplotlib.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
})

# Reproducibility
SEED = 42
rng = np.random.default_rng(SEED)

print('Setup complete.')

In [ ]:
# Import the gdo package
from gdo.optimizers import BatchGD, StochasticGD, MiniBatchGD, MomentumSGD
from gdo.landscapes import QuadraticSurface, Rosenbrock, Beale, Himmelblau, LandscapePlotter

print('gdo imports successful.')

---
## Step 2 — Quadratic Surface (Convex, Ill-Conditioned)

$$f(x, y) = a x^2 + b y^2$$

With $a=1, b=10$: the surface is 10× steeper in the $y$-direction.
This **ill-conditioning** causes plain gradient descent to oscillate across the steep axis,
which is why momentum and adaptive methods exist.

**Global minimum:** $(0, 0)$, $f=0$

In [ ]:
surface = QuadraticSurface(a=1.0, b=10.0)
start = (-2.5, 2.5)
N_STEPS = 150

optimizers_quad = [
    BatchGD(lr=0.08),
    StochasticGD(lr=0.08, noise_scale=0.1, seed=SEED),
    MiniBatchGD(lr=0.08, batch_size=32, noise_scale=0.04, seed=SEED),
    MomentumSGD(lr=0.08, momentum=0.9, noise_scale=0.04, seed=SEED),
]

for opt in optimizers_quad:
    surface.run_optimizer(opt, start=start, n_steps=N_STEPS)
    print(f'{opt.name:<30} final_loss={opt.loss_history[-1]:.6f}  steps={opt.n_steps}')

In [ ]:
fig = LandscapePlotter.contour(
    surface=surface,
    optimizers=optimizers_quad,
    log_scale=True,
    figsize=(9, 7),
    title='Quadratic Surface — Optimizer Trajectories',
)
plt.show()

In [ ]:
fig = LandscapePlotter.convergence_curves(
    optimizers=optimizers_quad,
    title='Quadratic Surface — Convergence (loss vs step)',
)
plt.show()

### What to observe
- **Batch GD** oscillates between the walls of the elongated valley
  — large updates in the steep $y$-direction overshoot repeatedly.
- **SGD** has even noisier oscillations due to per-sample gradient variance.
- **Mini-Batch GD** oscillates less than SGD but still more than Batch GD.
- **Momentum** accumulates velocity in the $x$-direction (consistent gradient)
  while dampening oscillations in $y$ — reaches the minimum much faster.

---
## Step 3 — Rosenbrock Function (Non-Convex, Banana Valley)

$$f(x, y) = (1-x)^2 + 100(y-x^2)^2$$

**Global minimum:** $(1, 1)$, $f=0$

The loss surface has a curved, narrow valley (the "banana").  
Finding the minimum requires navigating a path that changes curvature direction — a classic optimizer challenge.

This surface reveals:
- Batch GD gets stuck at the top of the valley wall for many steps.
- Momentum navigates the valley floor more efficiently.
- Starting point matters — the trajectory depends heavily on initialization.

In [ ]:
rosen = Rosenbrock()
start_rosen = (-1.5, 1.0)
N_ROSEN = 500

optimizers_rosen = [
    BatchGD(lr=0.001),
    StochasticGD(lr=0.001, noise_scale=0.005, seed=SEED),
    MomentumSGD(lr=0.001, momentum=0.9, noise_scale=0.002, seed=SEED),
]

for opt in optimizers_rosen:
    rosen.run_optimizer(opt, start=start_rosen, n_steps=N_ROSEN)
    print(f'{opt.name:<30} final_loss={opt.loss_history[-1]:.4f}  steps={opt.n_steps}')

In [ ]:
fig = LandscapePlotter.trajectory_comparison(
    surface=rosen,
    optimizers=optimizers_rosen,
    figsize=(14, 5),
)
plt.show()

In [ ]:
fig = LandscapePlotter.convergence_curves(
    optimizers=optimizers_rosen,
    title='Rosenbrock — Convergence (log scale)',
    log_scale=True,
)
plt.show()

---
## Step 4 — Himmelblau Function (Multi-Modal)

$$f(x, y) = (x^2 + y - 11)^2 + (x + y^2 - 7)^2$$

**Four global minima**, all with $f=0$:
- $(3.0, 2.0)$
- $(-2.805, 3.131)$
- $(-3.779, -3.283)$
- $(3.584, -1.848)$

**Key insight:** The starting point determines which minimum the optimizer converges to.
This is a fundamental property of gradient descent on non-convex surfaces — it finds
a local minimum near the initialization, not necessarily the global one.

In [ ]:
himmel = Himmelblau()

# Four different starting points — each converges to a different minimum
starts = [(0.0, 0.0), (-2.0, 3.5), (-4.0, -2.0), (4.0, -2.0)]
opts_himmel = []

for i, start in enumerate(starts):
    opt = BatchGD(lr=0.01)
    himmel.run_optimizer(opt, start=start, n_steps=300)
    traj = opt.trajectory
    end = traj[-1]
    print(f'Start={start}  →  End=({end[0]:.3f}, {end[1]:.3f})  loss={opt.loss_history[-1]:.4f}')
    opts_himmel.append(opt)

In [ ]:
# Show all four trajectories on a single contour plot
fig = LandscapePlotter.contour(
    surface=himmel,
    optimizers=opts_himmel,
    log_scale=True,
    figsize=(9, 8),
    title='Himmelblau — 4 Starting Points → 4 Different Minima',
    mark_optimum=False,
)
# Mark all four minima
ax = fig.axes[0]
for x, y in himmel.MINIMA:
    ax.plot(x, y, '*', color='gold', markersize=14, zorder=10)
plt.show()

---
## Step 5 — Side-by-Side Convergence Comparison

Same starting point. Same surface (Quadratic ill-conditioned).  
All four variants plotted together.

In [ ]:
# Summary table
print(f'{'Optimizer':<30} {'Steps to <0.01':<20} {'Final Loss':<15}')
print('-' * 65)
for opt in optimizers_quad:
    # Find first step where loss < 0.01
    conv_step = next(
        (i for i, l in enumerate(opt.loss_history) if l < 0.01),
        None
    )
    conv_str = str(conv_step) if conv_step is not None else 'not reached'
    print(f'{opt.name:<30} {conv_str:<20} {opt.loss_history[-1]:<15.6f}')

---
## Key Takeaways

1. **Batch GD** is stable but oscillates on ill-conditioned surfaces and  
   gets trapped on the Rosenbrock plateau.

2. **SGD** adds noise that helps escape local minima but never cleanly converges.

3. **Mini-Batch GD** is the best of both worlds — reduced variance from batching,  
   speed from not using the full dataset. This is what `torch.optim.SGD` uses in practice.

4. **Momentum** is the first adaptive technique — it doesn't change the learning rate  
   but changes the *direction* of updates by accumulating velocity.  
   On ill-conditioned surfaces, it reduces oscillation and accelerates convergence  
   by up to an order of magnitude.

5. **Initialization matters** — on non-convex surfaces like Himmelblau, the starting  
   point determines which local minimum gradient descent finds.

**Next:** Notebook 2 — Adaptive Optimizers (RMSProp, Adam, AdamW, Lion)  
These go further: instead of just changing update *direction*, they also  
adapt the *magnitude* per parameter.